# **Software Defect Prediction Using Machine Learning**

**1. Importing Required Libraries**

In [ ]:
# Import required libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt


from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score


from sklearn.preprocessing import StandardScaler


from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


from xgboost import XGBClassifier


from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix


import joblib

**2. Loading the Dataset**

In [ ]:
# Load datasets

train = pd.read_csv("/content/train.csv")

test = pd.read_csv("/content/test.csv")


print("Train shape:", train.shape)

print("Test shape:", test.shape)

**3. Understanding the Dataset**

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train.describe()

In [ ]:
train.columns

In [ ]:
train["defects"].value_counts()

In [ ]:
# Convert target values into binary format

train["defects"] = train["defects"].astype(int)

**4. Exploratory Data Analysis (EDA)**

In [ ]:
train.isnull().sum()
test.isnull().sum()

In [ ]:
# Replace missing values with column mean

train = train.fillna(train.mean())

test = test.fillna(test.mean())

In [ ]:
train["defects"].value_counts().plot(
    kind="bar"
)

plt.xlabel("Defect Class")

plt.ylabel("Count")

plt.title("Defect Distribution")

plt.show()

In [ ]:
correlation = train.corr()["defects"].sort_values()

correlation

In [ ]:
correlation.tail(10)

**5. Data Preprocessing**

In [ ]:
X = train.drop(
    "defects",
    axis=1
)


y = train["defects"]

In [ ]:
test_data = test.copy()

**6. Splitting Data for Training and Testing**

In [ ]:
# Stratified split keeps defect ratio similar in both sets

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print(X_train.shape)

print(X_valid.shape)

In [ ]:
scaler = StandardScaler()


X_train_scaled = scaler.fit_transform(
    X_train
)


X_valid_scaled = scaler.transform(
    X_valid
)

In [ ]:
X_train_scaled.shape

**7. Training Machine Learning Models**

In [ ]:
# Comparing different classification models

models = {

    "Logistic Regression": LogisticRegression(
        max_iter=1000
    ),


    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),


    "XGBoost": XGBClassifier(
        random_state=42
    )

}

**8. Model Evaluation Using Performance Metrics**

In [ ]:
def evaluate_model(name, model):

    model.fit(
        X_train_scaled,
        y_train
    )


    prediction = model.predict(
        X_valid_scaled
    )


    probability = model.predict_proba(
        X_valid_scaled
    )[:,1]


    accuracy = accuracy_score(
        y_valid,
        prediction
    )


    precision = precision_score(
        y_valid,
        prediction
    )


    recall = recall_score(
        y_valid,
        prediction
    )


    f1 = f1_score(
        y_valid,
        prediction
    )


    roc_auc = roc_auc_score(
        y_valid,
        probability
    )


    return [
        name,
        accuracy,
        precision,
        recall,
        f1,
        roc_auc
    ]

In [ ]:
results = []


for name, model in models.items():

    result = evaluate_model(
        name,
        model
    )

    results.append(result)

**9. Comparing Model Performance**

In [ ]:
results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ]
)


results_df

**10. Selecting the Best Performing Model**

In [ ]:
results_df.sort_values(
    by="F1 Score",
    ascending=False
)

In [ ]:
# Final selected model based on F1 score

best_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)


best_model.fit(
    X_train_scaled,
    y_train
)

**11. Evaluating the Selected Model**

In [ ]:
prediction = best_model.predict(
    X_valid_scaled
)


probability = best_model.predict_proba(
    X_valid_scaled
)[:,1]

In [ ]:
print(
    "Accuracy:",
    accuracy_score(
        y_valid,
        prediction
    )
)


print(
    "Precision:",
    precision_score(
        y_valid,
        prediction
    )
)


print(
    "Recall:",
    recall_score(
        y_valid,
        prediction
    )
)


print(
    "F1 Score:",
    f1_score(
        y_valid,
        prediction
    )
)


print(
    "ROC-AUC:",
    roc_auc_score(
        y_valid,
        probability
    )
)

In [ ]:
print(
    classification_report(
        y_valid,
        prediction
    )
)

In [ ]:
import seaborn as sns


cm = confusion_matrix(
    y_valid,
    prediction
)


plt.figure(figsize=(6,4))


sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)


plt.xlabel(
    "Predicted"
)


plt.ylabel(
    "Actual"
)


plt.title(
    "Confusion Matrix"
)


plt.show()

**12. Cross-Validation of the Selected Model**

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
cv_model = RandomForestClassifier(
    n_estimators=50,
    random_state=42
)


f1_scores = cross_val_score(
    cv_model,
    X,
    y,
    cv=cv,
    scoring="f1"
)


print(f1_scores)


print(
    "Average F1 Score:",
    f1_scores.mean()
)

In [ ]:
auc_scores = cross_val_score(
    best_model,
    X,
    y,
    cv=cv,
    scoring="roc_auc"
)


print(
    auc_scores
)


print(
    "Average ROC-AUC:",
    auc_scores.mean()
)

**13. Feature Importance Analysis**

In [ ]:
importance = pd.DataFrame(
    {
        "Feature": X.columns,
        "Importance": best_model.feature_importances_
    }
)


importance = importance.sort_values(
    by="Importance",
    ascending=False
)


importance.head(10)

In [ ]:
importance.head(10).plot(
    x="Feature",
    y="Importance",
    kind="bar",
    figsize=(10,5)
)


plt.title(
    "Top Features Affecting Software Defects"
)


plt.ylabel(
    "Importance"
)


plt.show()

**14. Training the Final Model**

In [ ]:
# Scaling complete training data

final_scaler = StandardScaler()


X_full_scaled = final_scaler.fit_transform(
    X
)


test_scaled = final_scaler.transform(
    test_data
)

In [ ]:
# Train final model using complete training data

final_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)


final_model.fit(
    X_full_scaled,
    y
)

In [ ]:
test_prediction = final_model.predict(
    test_scaled
)


test_probability = final_model.predict_proba(
    test_scaled
)[:,1]

In [ ]:
prediction_report = pd.DataFrame()


prediction_report["id"] = test_data["id"]


prediction_report["Defect Probability (%)"] = (
    test_probability * 100
).round(2)


prediction_report["Prediction"] = test_prediction


prediction_report.head()

In [ ]:
prediction_report["Prediction"] = prediction_report[
    "Prediction"
].replace(
    {
        0:"No Defect",
        1:"Defect"
    }
)


prediction_report.head()

In [ ]:
def risk_level(value):

    if value < 30:
        return "Low Risk"

    elif value < 70:
        return "Medium Risk"

    else:
        return "High Risk"

In [ ]:
prediction_report["Risk Level"] = prediction_report[
    "Defect Probability (%)"
].apply(
    risk_level
)


prediction_report.head()

In [ ]:
prediction_report["Quality Score"] = (
    100 - prediction_report["Defect Probability (%)"]
).round(2)


prediction_report.head()

In [ ]:
def recommendation(level):

    if level == "High Risk":
        return "Needs immediate code review and testing"

    elif level == "Medium Risk":
        return "Review module and perform additional testing"

    else:
        return "Module quality looks good"


prediction_report["Recommendation"] = prediction_report[
    "Risk Level"
].apply(
    recommendation
)


prediction_report.head()

In [ ]:
prediction_report.head(10)

**15. Saving the Trained Model and Scaler**

In [ ]:
import joblib


joblib.dump(
    final_model,
    "software_defect_model.pkl"
)


joblib.dump(
    final_scaler,
    "scaler.pkl"
)


print("Model and scaler saved successfully")

In [ ]:
import os

os.listdir()

**16. User CSV Upload for Prediction**

In [ ]:
from google.colab import files


uploaded = files.upload()

**17. Loading User Input Data**

In [ ]:
import os


file_name = list(uploaded.keys())[0]


user_data = pd.read_excel(
    file_name
)

user_data.head()

**18. Validating User Input Data**

In [ ]:
# Check whether required features are present

required_features = X.columns.tolist()


missing_columns = set(required_features) - set(user_data.columns)


if len(missing_columns) > 0:

    print(
        "Missing columns:",
        missing_columns
    )

else:

    print(
        "CSV format is correct"
    )

**19. Preprocessing User Input**

In [ ]:
user_ids = user_data["id"]


user_features = user_data[
    required_features
]

In [ ]:
user_scaled = final_scaler.transform(
    user_features
)

**20. Predicting Software Defects**

In [ ]:
user_prediction = final_model.predict(
    user_scaled
)


user_probability = final_model.predict_proba(
    user_scaled
)[:,1]

**21. Generating the Software Defect Prediction Report**

In [ ]:
user_report = pd.DataFrame()


user_report["id"] = user_ids


user_report["Defect Probability (%)"] = (
    user_probability * 100
).round(2)


user_report["Prediction"] = user_prediction


user_report.head()

In [ ]:
user_report["Prediction"] = user_report[
    "Prediction"
].replace(
    {
        0:"No Defect",
        1:"Defect"
    }
)


user_report.head()

In [ ]:
user_report["Risk Level"] = user_report[
    "Defect Probability (%)"
].apply(
    risk_level
)


user_report.head()

In [ ]:
user_report["Quality Score"] = (
    100 - user_report["Defect Probability (%)"]
).round(2)


user_report.head()

In [ ]:
user_report["Recommendation"] = user_report[
    "Risk Level"
].apply(
    recommendation
)


user_report.head()

In [ ]:
user_report

**22. Exporting Prediction Results**

In [ ]:
user_report.to_csv(
    "defect_prediction_result.csv",
    index=False
)


files.download(
    "defect_prediction_result.csv"
)